# Hardware Emit Pass
The `emit_verilog` transform pass generates a top-level RTL file and testbench file according to the `MaseGraph`, which includes a hardware implementation of each layer in the network. This top-level file instantiates modules from the `components` library in MASE and/or modules generated using [HLS](https://en.wikipedia.org/wiki/High-level_synthesis), when internal components are not available. The hardware can then be simulated using [Verilator](https://www.veripool.org/verilator/), or deployed on an FPGA.

First, add Machop to your system PATH (if you haven't already done so) and import the required libraries.

In [ ]:
import os, sys
import torch
torch.manual_seed(0)

from chop.ir.graph.mase_graph import MaseGraph

from chop.passes.graph.analysis import (
    init_metadata_analysis_pass,
    add_common_metadata_analysis_pass,
    add_hardware_metadata_analysis_pass,
    add_software_metadata_analysis_pass,
    report_node_type_analysis_pass,
)

from chop.passes.graph.transforms import (
    emit_verilog_top_transform_pass,
    emit_internal_rtl_transform_pass,
    emit_bram_transform_pass,
    emit_cocotb_transform_pass,
    quantize_transform_pass,
)

from chop.tools.logger import set_logging_verbosity

set_logging_verbosity("debug")

import toml
import torch
import torch.nn as nn

# TO DO: remove
import os
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ["PATH"]
!verilator

INFO     Set logging level to debug


Usage:
        verilator --help
        verilator --version
        verilator --binary -j 0 [options] [source_files.v]... [opt_c_files.cpp/c/cc/a/o/so]
        verilator --cc [options] [source_files.v]... [opt_c_files.cpp/c/cc/a/o/so]
        verilator --sc [options] [source_files.v]... [opt_c_files.cpp/c/cc/a/o/so]
        verilator --lint-only -Wall [source_files.v]...



# CNN Model

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

class SingleConvNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.fc1 = nn.Linear(32, 4)

    def forward(self, x):
        x = self.maxpool(x)
        x = torch.flatten(x, start_dim=1, end_dim=-1)
        x = self.fc1(x)
        return x


In [ ]:
cnn = SingleConvNet()
mg = MaseGraph(model=cnn)

# Provide a dummy input for the graph so it can use for tracing
batch_size = 1
x = torch.randn((batch_size,2, 8, 8))
dummy_in = {"x": x}


In [ ]:

mg, _ = init_metadata_analysis_pass(mg, None)


In [ ]:

mg, _ = add_common_metadata_analysis_pass(
    mg, {"dummy_in": dummy_in, "add_value": False}
)



DEBUG    graph():
    %x : [num_users=1] = placeholder[target=x]
    %maxpool : [num_users=1] = call_module[target=maxpool](args = (%x,), kwargs = {})
    %flatten : [num_users=1] = call_function[target=torch.flatten](args = (%maxpool,), kwargs = {start_dim: 1, end_dim: -1})
    %fc1 : [num_users=1] = call_module[target=fc1](args = (%flatten,), kwargs = {})
    return fc1


Hellos in add_common_metadata
sigoyi in add_common_metadata
nihhhhhhhhh in graph_iterator_for_mase_ops in add_common_metadata
graph_model:  GraphModule(
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32, out_features=4, bias=True)
)



def forward(self, x):
    maxpool = self.maxpool(x);  x = None
    flatten = torch.flatten(maxpool, start_dim = 1, end_dim = -1);  maxpool = None
    fc1 = self.fc1(flatten);  flatten = None
    return fc1
    
# To see more debug info, please use `graph_module.print_readable()`
wocao in add_common_metadata
graph_iterator_for_metadata:  GraphModule(
  (maxpool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=32, out_features=4, bias=True)
)



def forward(self, x):
    maxpool = self.maxpool(x);  x = None
    flatten = torch.flatten(maxpool, start_dim = 1, end_dim = -1);  maxpool = None
    fc1 = self.fc1(flatten);  flatten = None


# Quantize

In [ ]:
config_file = os.path.join(
    os.path.abspath(""),
    "..",
    "..",
    "configs",
    "tests",
    "quantize",
    "fixed.toml",
)
# Fixed.toml used to quantize the model
with open(config_file, "r") as f:
    quan_args = toml.load(f)["passes"]["quantize"]
mg, _ = quantize_transform_pass(mg, quan_args)

_ = report_node_type_analysis_pass(mg)

# Update the metadata
for node in mg.fx_graph.nodes:
    for arg, arg_info in node.meta["mase"]["common"]["args"].items():
        if isinstance(arg_info, dict):
            arg_info["type"] = "fixed"
            arg_info["precision"] = [8, 5]
    for result, result_info in node.meta["mase"]["common"]["results"].items():
        if isinstance(result_info, dict):
            result_info["type"] = "fixed"
            result_info["precision"] = [8, 5]

INFO     Inspecting graph [add_common_node_type_analysis_pass]
INFO     
Node name    Fx Node op     Mase type            Mase op      Value type
-----------  -------------  -------------------  -----------  ------------
x            placeholder    placeholder          placeholder  NA
maxpool      call_module    module_related_func  max_pool2d   float
flatten      call_function  implicit_func        flatten      float
fc1          call_module    module_related_func  linear       fixed
output       output         output               output       NA


placeholder
max_pool2d
flatten
node_config:  {'name': 'fixed', 'data_in_width': 8, 'data_in_frac_width': 5, 'weight_width': 8, 'weight_frac_width': 5, 'bias_width': 8, 'bias_frac_width': 5, 'data_out_width': 8, 'data_out_frac_width': 5, 'floor': True}
output


Max Pooling

In [ ]:
mg, _ = add_hardware_metadata_analysis_pass(mg)

mase_op: max_pool2d
2222
mase_op: linear
2222
 I changed max_parallelism add_hardware_metadata_analysis_pass
vp:  {}
arg:  OrderedDict([('data_in_0', {'shape': [1, 2, 8, 8], 'torch_dtype': torch.float32, 'type': 'fixed', 'precision': [8, 5]})])
arg_info in add_verilog_param:  {'shape': [1, 2, 8, 8], 'torch_dtype': torch.float32, 'type': 'fixed', 'precision': [8, 5]}
arg in add_verilog_param: data_in_0
vp after for:  {'DATA_IN_0_PRECISION_0': 8, 'DATA_IN_0_PRECISION_1': 5, 'DATA_IN_0_TENSOR_SIZE_DIM_0': 8, 'DATA_IN_0_PARALLELISM_DIM_0': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_1': 8, 'DATA_IN_0_PARALLELISM_DIM_1': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_2': 2, 'DATA_IN_0_PARALLELISM_DIM_2': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_3': 1, 'DATA_IN_0_PARALLELISM_DIM_3': 1}
result in add_verilog_param: data_out_0
result_info in add_verilog_param: {'type': 'fixed', 'precision': [8, 5], 'shape': [1, 2, 4, 4], 'torch_dtype': torch.float32}
vp:  {}
arg:  OrderedDict([('data_in_0', {'shape': [1, 32], 'torch_dtype': torch.floa

# Hardware Metapass

In [ ]:
mg, _ = add_hardware_metadata_analysis_pass(mg)
# for node in mg.nodes:
#         mase_op = node.meta["mase"]["common"]["mase_op"]
#         print ("node.op:", node.op)
#         print ('mase_op:', mase_op)
#         print ("common:",node.meta["mase"]["common"])
#         print ("hardware:",node.meta["mase"]["hardware"])

# for node in mg.fx_graph.nodes:
#         if node.meta["mase"].parameters["hardware"]["is_implicit"]:
#             continue
#         # Only modules have internal parameters
#         if node.meta["mase"].module is None:
#             continue
#         # print (node.meta["mase"].parameters["hardware"])
#         # Only checks the hardware data that contains the key toolchain
#         if "INTERNAL" in node.meta["mase"].parameters["hardware"]["toolchain"]:
#                 for param_name, parameter in node.meta["mase"].module.named_parameters():
#                         print ("param_name in CNN.jynb:",param_name)
#                         print ("parameter in CNN.jynb:", parameter)

# """
# weights and bias in the Conv/linear in Maze
# param_data = node.meta["mase"].module.get_parameter(param_name).data
# print ("param_data: ", param_data)
# """


mase_op: max_pool2d
2222
mase_op: linear
2222
 I changed max_parallelism add_hardware_metadata_analysis_pass
vp:  {}
arg:  OrderedDict([('data_in_0', {'shape': [1, 2, 8, 8], 'torch_dtype': torch.float32, 'type': 'fixed', 'precision': [8, 5]})])
arg_info in add_verilog_param:  {'shape': [1, 2, 8, 8], 'torch_dtype': torch.float32, 'type': 'fixed', 'precision': [8, 5]}
arg in add_verilog_param: data_in_0
vp after for:  {'DATA_IN_0_PRECISION_0': 8, 'DATA_IN_0_PRECISION_1': 5, 'DATA_IN_0_TENSOR_SIZE_DIM_0': 8, 'DATA_IN_0_PARALLELISM_DIM_0': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_1': 8, 'DATA_IN_0_PARALLELISM_DIM_1': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_2': 2, 'DATA_IN_0_PARALLELISM_DIM_2': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_3': 1, 'DATA_IN_0_PARALLELISM_DIM_3': 1}
result in add_verilog_param: data_out_0
result_info in add_verilog_param: {'type': 'fixed', 'precision': [8, 5], 'shape': [1, 2, 4, 4], 'torch_dtype': torch.float32}
vp:  {}
arg:  OrderedDict([('data_in_0', {'shape': [1, 32], 'torch_dtype': torch.floa

# Emit for SV file

In [ ]:
# base_dir = "/home/kai/mase/src/mase_components"
# dependency_path = os.path.join(base_dir, "pooling_layers", "rtl")
# dependency_path = os.path.abspath(dependency_path)

# for node in mg.fx_graph.nodes:
#     hw_params = node.meta["mase"]["hardware"]
#     if node.name == "maxpool":
#         hw_params = node.meta["mase"]["hardware"]
#         hw_params["toolchain"] = "INTERNAL_RTL" 
#         hw_params["module"] = "max_pooling_2d"

# for node in mg.fx_graph.nodes:
#     # Now check based on the hardware module field instead of node name
#     if node.meta["mase"]["hardware"].get("module") == "max_pooling_2d":
#         hw_meta = node.meta["mase"]["hardware"]
        # hw_meta["interface"] = {}
        
        # hw_meta["dependence_files"] = [
        #     os.path.join(dependency_path, "fifo.sv"),
        #     os.path.join(dependency_path, "max_pooling_2d.sv"),
        #     os.path.join(dependency_path, "pool_window.sv")
        # ]
        # hw_meta["verilog_param"] = {
        #     "DATA_IN_0_PRECISION_0": 8,
        #     "DATA_IN_0_PRECISION_1": 5,
        #     "DATA_OUT_0_PRECISION_0": 8,
        #     "DATA_OUT_0_PRECISION_1": 5,
        #     "POOL_SIZE": 2,
            # "DATA_IN_0_PARALLELISM_DIM_0": 2,
            # "DATA_IN_0_PARALLELISM_DIM_1": 2,
            # "DATA_IN_0_PARALLELISM_DIM_2": 2,
            # "DATA_IN_0_PARALLELISM_DIM_3": 1,
            # "DATA_OUT_0_PARALLELISM_DIM_0": 2,
            # "DATA_OUT_0_PARALLELISM_DIM_1": 2,
            # "DATA_OUT_0_PARALLELISM_DIM_2": 2,
            # "DATA_OUT_0_PARALLELISM_DIM_3": 1,
            # "FIFO_DEPTH" : 4,
        # }
        # node.meta["mase"]["hardware"] = hw_meta


In [ ]:
for node in mg.fx_graph.nodes:
    hardware_meta = node.meta["mase"]["hardware"]
    print(f"Node {node.name} hardware metadata: {hardware_meta}")

Node x hardware metadata: {'is_implicit': True, 'device_id': 0, 'max_parallelism': [2, 2, 2, 2]}
Node maxpool hardware metadata: {'is_implicit': False, 'device_id': -1, 'interface': {}, 'toolchain': 'INTERNAL_RTL', 'module': 'max_pooling_2d', 'dependence_files': ['memory/rtl/fifo.sv', 'pooling_layers/rtl/max_pooling_2d.sv', 'pooling_layers/rtl/pool_window.sv'], 'max_parallelism': [2, 2, 2, 2], 'verilog_param': {'DATA_IN_0_PRECISION_0': 8, 'DATA_IN_0_PRECISION_1': 5, 'DATA_IN_0_TENSOR_SIZE_DIM_0': 8, 'DATA_IN_0_PARALLELISM_DIM_0': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_1': 8, 'DATA_IN_0_PARALLELISM_DIM_1': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_2': 2, 'DATA_IN_0_PARALLELISM_DIM_2': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_3': 1, 'DATA_IN_0_PARALLELISM_DIM_3': 1, 'DATA_OUT_0_PRECISION_0': 8, 'DATA_OUT_0_PRECISION_1': 5, 'DATA_OUT_0_TENSOR_SIZE_DIM_0': 4, 'DATA_OUT_0_PARALLELISM_DIM_0': 2, 'DATA_OUT_0_TENSOR_SIZE_DIM_1': 4, 'DATA_OUT_0_PARALLELISM_DIM_1': 2, 'DATA_OUT_0_TENSOR_SIZE_DIM_2': 2, 'DATA_OUT_0_PARALLELISM_

In [ ]:
mg, _ = emit_verilog_top_transform_pass(mg)
mg, _ = emit_internal_rtl_transform_pass(mg)

INFO     Emitting Verilog...
INFO     Emitting internal components...


Parameter_map in VerilogEmitter: {'maxpool_DATA_IN_0_PRECISION_0': 8, 'maxpool_DATA_IN_0_PRECISION_1': 5, 'maxpool_DATA_IN_0_TENSOR_SIZE_DIM_0': 8, 'maxpool_DATA_IN_0_PARALLELISM_DIM_0': 2, 'maxpool_DATA_IN_0_TENSOR_SIZE_DIM_1': 8, 'maxpool_DATA_IN_0_PARALLELISM_DIM_1': 2, 'maxpool_DATA_IN_0_TENSOR_SIZE_DIM_2': 2, 'maxpool_DATA_IN_0_PARALLELISM_DIM_2': 2, 'maxpool_DATA_IN_0_TENSOR_SIZE_DIM_3': 1, 'maxpool_DATA_IN_0_PARALLELISM_DIM_3': 1, 'maxpool_DATA_OUT_0_PRECISION_0': 8, 'maxpool_DATA_OUT_0_PRECISION_1': 5, 'maxpool_DATA_OUT_0_TENSOR_SIZE_DIM_0': 4, 'maxpool_DATA_OUT_0_PARALLELISM_DIM_0': 2, 'maxpool_DATA_OUT_0_TENSOR_SIZE_DIM_1': 4, 'maxpool_DATA_OUT_0_PARALLELISM_DIM_1': 2, 'maxpool_DATA_OUT_0_TENSOR_SIZE_DIM_2': 2, 'maxpool_DATA_OUT_0_PARALLELISM_DIM_2': 2, 'maxpool_DATA_OUT_0_TENSOR_SIZE_DIM_3': 1, 'maxpool_DATA_OUT_0_PARALLELISM_DIM_3': 1, 'fc1_DATA_IN_0_PRECISION_0': 8, 'fc1_DATA_IN_0_PRECISION_1': 5, 'fc1_DATA_IN_0_TENSOR_SIZE_DIM_0': 32, 'fc1_DATA_IN_0_PARALLELISM_DIM_0': 2,

# Memory

In [ ]:
mg, _ = emit_bram_transform_pass(mg)

"""
  param_data:  tensor([[[[-0.1946,  0.2865,  0.1487],
          [ 0.1616,  0.0175, -0.1709],
          [ 0.0564, -0.3112, -0.2409]]],


        [[[-0.1718,  0.2103,  0.1954],
          [-0.1478, -0.0120,  0.2132],
          [ 0.3314,  0.1323,  0.0450]]]])
  This would be weight, as I defined conv as (2,1,3,3)
  The kernel filter size is 3x3, and we have 2 filters, thus 9x2=18 elements
  Bias: depends on number of filter-> here 2 filters, thus 2 bias elements,
    as bias added after filter multiplies with section of pixel
"""

INFO     Emitting BRAM...
DEBUG    Emitting DAT file for node: fc1, parameter: weight
DEBUG    ROM module weight successfully written into /home/kai/.mase/top/hardware/rtl/fc1_weight_source.sv
DEBUG    Init data weight successfully written into /home/kai/.mase/top/hardware/rtl/fc1_weight_rom.dat
DEBUG    Emitting DAT file for node: fc1, parameter: bias
DEBUG    ROM module bias successfully written into /home/kai/.mase/top/hardware/rtl/fc1_bias_source.sv


1 in emit_bram_transform_pass
/home/kai/.mase/top/hardware/rtl
1 in emit_bram_transform_pass
/home/kai/.mase/top/hardware/rtl
param_name in emit_bram_handshake: weight
out_size: 4
out_size in emit_parameters_in_mem_internal: 4
fc1
verilog_param_name weight
total_size: 128
shape: [4, 32]
 node.meta[mase].parameters[hardware][verilog_param] {'DATA_IN_0_PRECISION_0': 8, 'DATA_IN_0_PRECISION_1': 5, 'DATA_IN_0_TENSOR_SIZE_DIM_0': 32, 'DATA_IN_0_PARALLELISM_DIM_0': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_1': 1, 'DATA_IN_0_PARALLELISM_DIM_1': 1, 'WEIGHT_PRECISION_0': 8, 'WEIGHT_PRECISION_1': 5, 'WEIGHT_TENSOR_SIZE_DIM_0': 32, 'WEIGHT_PARALLELISM_DIM_0': 2, 'WEIGHT_TENSOR_SIZE_DIM_1': 4, 'WEIGHT_PARALLELISM_DIM_1': 2, 'BIAS_PRECISION_0': 8, 'BIAS_PRECISION_1': 5, 'BIAS_TENSOR_SIZE_DIM_0': 4, 'BIAS_PARALLELISM_DIM_0': 2, 'BIAS_TENSOR_SIZE_DIM_1': 1, 'BIAS_PARALLELISM_DIM_1': 1, 'DATA_OUT_0_PRECISION_0': 8, 'DATA_OUT_0_PRECISION_1': 5, 'DATA_OUT_0_TENSOR_SIZE_DIM_0': 4, 'DATA_OUT_0_PARALLELISM_DIM_0': 2, 

DEBUG    Init data bias successfully written into /home/kai/.mase/top/hardware/rtl/fc1_bias_rom.dat


fc1
verilog_param_name bias
total_size: 4
shape: [1, 4]
 node.meta[mase].parameters[hardware][verilog_param] {'DATA_IN_0_PRECISION_0': 8, 'DATA_IN_0_PRECISION_1': 5, 'DATA_IN_0_TENSOR_SIZE_DIM_0': 32, 'DATA_IN_0_PARALLELISM_DIM_0': 2, 'DATA_IN_0_TENSOR_SIZE_DIM_1': 1, 'DATA_IN_0_PARALLELISM_DIM_1': 1, 'WEIGHT_PRECISION_0': 8, 'WEIGHT_PRECISION_1': 5, 'WEIGHT_TENSOR_SIZE_DIM_0': 32, 'WEIGHT_PARALLELISM_DIM_0': 2, 'WEIGHT_TENSOR_SIZE_DIM_1': 4, 'WEIGHT_PARALLELISM_DIM_1': 2, 'BIAS_PRECISION_0': 8, 'BIAS_PRECISION_1': 5, 'BIAS_TENSOR_SIZE_DIM_0': 4, 'BIAS_PARALLELISM_DIM_0': 2, 'BIAS_TENSOR_SIZE_DIM_1': 1, 'BIAS_PARALLELISM_DIM_1': 1, 'DATA_OUT_0_PRECISION_0': 8, 'DATA_OUT_0_PRECISION_1': 5, 'DATA_OUT_0_TENSOR_SIZE_DIM_0': 4, 'DATA_OUT_0_PARALLELISM_DIM_0': 2, 'DATA_OUT_0_TENSOR_SIZE_DIM_1': 1, 'DATA_OUT_0_PARALLELISM_DIM_1': 1}
out_size: 2
param_data in emit_parameters_in_dat_internal:  tensor([-0.0618, -0.1449, -0.0376,  0.0378])
param_data in emit_parameters_in_dat_internal after flatt

'\n  param_data:  tensor([[[[-0.1946,  0.2865,  0.1487],\n          [ 0.1616,  0.0175, -0.1709],\n          [ 0.0564, -0.3112, -0.2409]]],\n\n\n        [[[-0.1718,  0.2103,  0.1954],\n          [-0.1478, -0.0120,  0.2132],\n          [ 0.3314,  0.1323,  0.0450]]]])\n  This would be weight, as I defined conv as (2,1,3,3)\n  The kernel filter size is 3x3, and we have 2 filters, thus 9x2=18 elements\n  Bias: depends on number of filter-> here 2 filters, thus 2 bias elements,\n    as bias added after filter multiplies with section of pixel\n'

In [ ]:
mg, _ = emit_cocotb_transform_pass(mg)

INFO     Emitting testbench...


In [ ]:
from chop.actions import simulate

simulate(skip_build=False, skip_test=False)

INFO: Running command perl /home/kai/miniconda3/envs/mase/bin/verilator -cc --exe -Mdir /home/kai/mase/docs/labs/sim_build -DCOCOTB_SIM=1 --top-module top --vpi --public-flat-rw --prefix Vtop -o top -LDFLAGS '-Wl,-rpath,/home/kai/miniconda3/envs/mase/lib/python3.11/site-packages/cocotb/libs -L/home/kai/miniconda3/envs/mase/lib/python3.11/site-packages/cocotb/libs -lcocotbvpi_verilator' -Wno-fatal -Wno-lint -Wno-style --trace-fst --trace-structs --trace-depth 3 --stats -I/home/kai/.mase/top/hardware/rtl -I/home/kai/mase/src/mase_components/common/rtl -I/home/kai/mase/src/mase_components/vivado/rtl -I/home/kai/mase/src/mase_components/helper/rtl -I/home/kai/mase/src/mase_components/language_models/rtl -I/home/kai/mase/src/mase_components/systolic_arrays/rtl -I/home/kai/mase/src/mase_components/convolution_layers/rtl -I/home/kai/mase/src/mase_components/cast/rtl -I/home/kai/mase/src/mase_components/memory/rtl -I/home/kai/mase/src/mase_components/pooling_layers/rtl -I/home/kai/mase/src/mas

%Error: /home/kai/.mase/top/hardware/rtl/top.sv:240:33: Can't find definition of variable: 'flatten_data_out_0_valid'
                                                      : ... Suggested alternative: 'flatten_data_out_0_ready'
  240 | assign fc1_data_in_0_valid    = flatten_data_out_0_valid;
      |                                 ^~~~~~~~~~~~~~~~~~~~~~~~
%Error: /home/kai/.mase/top/hardware/rtl/top.sv:241:24: Can't find definition of variable: 'flatten_data_out_0'
                                                      : ... Suggested alternative: 'fc1_data_out_0'
  241 | assign fc1_data_in_0 = flatten_data_out_0;
      |                        ^~~~~~~~~~~~~~~~~~
%Error: Exiting due to 2 error(s)


SystemExit: Process 'perl' terminated with error 1